# Lesson 07 - পরিকল্পনা ডিজাইন প্যাটার্ন

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে AI এজেন্টদের জন্য **পরিকল্পনা ডিজাইন প্যাটার্ন** প্রদর্শন করে।
আপনি শিখবেন কীভাবে জটিল ভ্রমণ অনুরোধগুলিকে গঠনমূলক সাবটাস্কে ভাগ করতে হয়,
তাদের বিশেষজ্ঞ এজেন্টদের কাছে নিয়োগ দিতে হয়, এবং প্রাপ্ত পরিকল্পনাটি কার্যকর করতে হয় — সবই Pydantic মডেল দ্বারা পরিচালিত গঠিত আউটপুট ব্যবহার করে।


## সেটআপ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## কাজের বিশ্লেষণ

কাজের বিশ্লেষণ পরিকল্পনা ডিজাইন প্যাটার্নের মূল। একটি জটিল অনুরোধ সম্পূর্ণরূপে পরিচালনার জন্য একটি একক এজেন্টের পরিবর্তে, আমরা সমস্যাটিকে ছোট, সুসংজ্ঞায়িত **সাবটাস্ক** এ ভাগ করি। প্রতিটি সাবটাস্ক একটি বিশেষজ্ঞ এজেন্ট (যেমন, ফ্লাইট, হোটেল, ক্রিয়াকলাপ) কে স্পষ্ট অগ্রাধিকার এবং নির্ভরতার ক্রমে নির্ধারিত করা হয়।

এই পদ্ধতিটি কয়েকটি সুবিধা প্রদান করে:
- **সুস্পষ্টতা**: প্রতিটি সাবটাস্কের একটি একক দায়িত্ব থাকে।
- **সমান্তরালতা**: স্বাধীন সাবটাস্কগুলো সমান্তরালে চলতে পারে।
- **নির্ভরযোগ্যতা**: ব্যর্থতা পৃথক সাবটাস্কে সীমাবদ্ধ থাকে।
- **বাজেট ট্র্যাকিং**: খরচ প্রতি সাবটাস্ক অনুযায়ী अनुमानিত এবং সম্মিলিত হয়।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## কাঠামোগত আউটপুট সহ একটি পরিকল্পনা এজেন্ট তৈরি করা

পরিকল্পনা এজেন্ট একটি **ফ্রন্ট ডেস্ক সমন্বয়কারী** হিসাবে কাজ করে। উচ্চ-স্তরের একটি ভ্রমণের অনুরোধ দেওয়ার পর এটি একটি কাঠামোবদ্ধ `TravelPlan` তৈরি করে — অনুরোধটিকে উপকর্মে বিভক্ত করা, অগ্রাধিকার নির্ধারণ করা এবং নির্ভরশীলতাগুলি চিহ্নিত করা যাতে একটি কনসিয়ার্জ বা কার্যাদান স্তর কাজটি এগিয়ে নিতে পারে।


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## স্পেশালিস্ট টুলগুলোর মাধ্যমে একটি পরিকল্পনা বাস্তবায়ন করা

যখন ফ্রন্ট ডেস্ক এজেন্ট একটি গঠনমূলক পরিকল্পনা তৈরি করে, তখন **কনসিয়ার্জ এজেন্ট** এটি বাস্তবায়ন করে। প্রতিটি স্পেশালিস্ট টুল একটি উপকার্যের উর্ধ্বে কাজ করে (ফ্লাইট, হোটেল, কার্যক্রম)। কনসিয়ার্জ পরিকল্পনার উপকার্যগুলো নির্ভরশীলতার ক্রমে পর্যায়ক্রমে চালায় এবং প্রতিটির জন্য উপযুক্ত টুলকে বরাদ্দ করে।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## সারাংশ

এই পাঠে আপনি AI এজেন্টদের জন্য **পরিকল্পনা নকশার প্যাটার্ন** শিখেছেন:

1. **কার্য বিভাজন** — একটি ফ্রন্ট ডেস্ক পরিকল্পনা এজেন্ট একটি জটিল ভ্রমণ অনুরোধকে কাঠামোবদ্ধ উপকার্যে ভাঙে, প্রতিটি উপকার্যকে অগ্রাধিকার এবং নির্ভরতাসহ একটি বিশেষজ্ঞ এজেন্টকে বরাদ্দ করে Pydantic মডেল ব্যবহার করে।
2. **কাঠামোবদ্ধ আউটপুট** — একটি `response_format` প্রদান করে এজেন্ট একটি validated `TravelPlan` অবজেক্ট ফেরত দেয় ফ্রি-ফর্ম টেক্সটের পরিবর্তে, যা নিচের ধাপের প্রক্রিয়াকরণকে নির্ভরযোগ্য করে তোলে।
3. **পরিকল্পনা সম্পাদন** — একটি কনসাইয়ার্জ এজেন্ট বিশেষজ্ঞ টুলস (`book_flight`, `reserve_hotel`, `book_activity`) ব্যবহার করে উপকার্যগুলোর মধ্য দিয়ে যায় পরিকল্পনাটি কার্যকর করতে এবং ফলাফল রিপোর্ট করতে।

এই প্যাটার্নটি *কী করতে হবে* (পরিকল্পনা) এবং *কিভাবে করতে হবে* (সম্পাদন) আলাদা করে, যা এজেন্টদের আরও মডুলার, পরীক্ষাযোগ্য এবং সহজে সম্প্রসারণযোগ্য করে তোলে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**বিস্তারিত বিবৃতি**:  
এই দলিলটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসম্ভব সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসঙ্গতি থাকতে পারে। মূল দলিলটির স্থানীয় ভাষার সংস্করণই কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুতর তথ্যের জন্য পেশাদার মানুষের অনুবাদের সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে কোনো ভুল বোঝাবুঝি বা অসঙ্গতির জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
